In [3]:
import geopandas as gpd
import os
import h3

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#Read the files
index_walkability = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability = index_walkability.to_crs(operation_crs)
zones_mmt = gpd.read_file(f'{input_file_path}/network_agreg/mmt/ZonesMMT.shp')
zones_mmt = zones_mmt.to_crs(operation_crs)

# Spatial join
segments_mmt = gpd.sjoin(index_walkability, zones_mmt, how="inner", predicate="within")


In [4]:
mmt_walk = segments_mmt.groupby("MMT_NO")["walkability_index"].mean().reset_index()

# Merge back with zones_mmt polygons
zones_mmt = zones_mmt.merge(mmt_walk, on="MMT_NO", how="left")


In [6]:
#Length-weighted mean: longer segments should contribute more.
#segments_mmt["length"] = segments_mmt.geometry.length
#mmt_walk = (segments_mmt["walkability_index"] * segments_mmt["length"]).groupby(segments_mmt["MMT_NO"]).sum() / segments_mmt.groupby("MMT_NO")["length"].sum()


In [7]:
#Drop nan values 
zones_mmt = zones_mmt.dropna(subset=["walkability_index"])

**Add h3**

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Polygon, MultiPolygon, box
import h3
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# -----------------------------------------------
# CONFIGURATION
# -----------------------------------------------
RES = 9
CRS_WGS84 = "EPSG:4326"

# -----------------------------------------------
# 1. Load and prepare the walkability data
# -----------------------------------------------
# Reload the original data to ensure we have proper CRS
index_walkability_orig = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability_orig = index_walkability_orig.to_crs(operation_crs)

print(f"Loaded {len(index_walkability_orig)} segments with CRS: {index_walkability_orig.crs}")

# Calculate centroids in projected coordinates first (more accurate for LineStrings)
centroids_projected = index_walkability_orig.geometry.centroid
print(f"Calculated {len(centroids_projected)} centroids in projected CRS")

# Convert centroids to WGS84 for H3
centroids_wgs84 = gpd.GeoSeries(centroids_projected, crs=operation_crs).to_crs(CRS_WGS84)

# Assign H3 IDs using accurate centroids
index_walkability_orig["h3_id"] = centroids_wgs84.apply(
    lambda pt: h3.latlng_to_cell(pt.y, pt.x, RES)
)

print(f"✅ Assigned H3 IDs to {len(index_walkability_orig)} segments")

# -----------------------------------------------
# 2. Aggregate walkability by H3 (mean per hexagon)
# -----------------------------------------------
h3_walkability = (
    index_walkability_orig.groupby("h3_id", as_index=False)
    .agg(walkability_mean=("walkability_index", "mean"),
         segment_count=("walkability_index", "size"))
)

print(f"✅ Created {len(h3_walkability)} H3 hexagons with walkability data")
print(f"Sample walkability values: {h3_walkability['walkability_mean'].head()}")
print(f"Walkability range: {h3_walkability['walkability_mean'].min():.3f} to {h3_walkability['walkability_mean'].max():.3f}")

# -----------------------------------------------
# 3. Add lat/lon coordinates for each H3 hexagon
# -----------------------------------------------
h3_coords = []
for h3_id in h3_walkability['h3_id']:
    lat, lon = h3.cell_to_latlng(h3_id)
    h3_coords.append({'h3_id': h3_id, 'lat': lat, 'lon': lon})

h3_coords_df = pd.DataFrame(h3_coords)
h3_complete = h3_walkability.merge(h3_coords_df, on='h3_id')

print(f"✅ Added coordinates to {len(h3_complete)} H3 hexagons")

# -----------------------------------------------
# 4. Create a complete grid and fill gaps with regression
# -----------------------------------------------
# Get all H3 neighbors to create a denser grid (optional)
all_h3_ids = set(h3_walkability['h3_id'])

# Add neighbors to create a more complete grid
for h3_id in list(all_h3_ids):
    neighbors = h3.grid_ring(h3_id, 1)  # Get direct neighbors
    all_h3_ids.update(neighbors)

print(f"Extended grid to {len(all_h3_ids)} hexagons (including neighbors)")

# Create complete grid
h3_coords_extended = []
for h3_id in all_h3_ids:
    lat, lon = h3.cell_to_latlng(h3_id)
    h3_coords_extended.append({'h3_id': h3_id, 'lat': lat, 'lon': lon})

h3_grid_complete = pd.DataFrame(h3_coords_extended)
h3_final = h3_grid_complete.merge(h3_walkability, on='h3_id', how='left')

# Fill missing values with regression
has_data = h3_final.dropna(subset=["walkability_mean"])
missing = h3_final[h3_final["walkability_mean"].isna()]

if len(missing) > 0 and len(has_data) > 2:
    print(f"Running regression: {len(has_data)} training points → {len(missing)} predictions")
    
    X_train = has_data[["lat", "lon"]].values
    y_train = has_data["walkability_mean"].values
    X_pred = missing[["lat", "lon"]].values

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_pred_s = scaler.transform(X_pred)

    reg = LinearRegression()
    reg.fit(X_train_s, y_train)
    y_pred = reg.predict(X_pred_s)

    # Fill missing values
    h3_final.loc[h3_final["walkability_mean"].isna(), "walkability_mean"] = y_pred
    h3_final.loc[h3_final["segment_count"].isna(), "segment_count"] = 0
    
    print(f"✅ Filled {len(missing)} missing values using linear regression")

# -----------------------------------------------
# 5. Convert H3 cells to polygons for mapping
# -----------------------------------------------
def h3_boundary_polygon(h):
    ring = h3.cell_to_boundary(h)
    # h3.cell_to_boundary returns [(lat, lon), ...] 
    # Shapely Polygon expects [(lon, lat), ...] for WGS84
    return Polygon([(lon, lat) for lat, lon in ring])

hex_polys = [h3_boundary_polygon(h) for h in h3_final["h3_id"]]

gdf_hex = gpd.GeoDataFrame(
    h3_final[["h3_id", "lat", "lon", "walkability_mean", "segment_count"]],
    geometry=hex_polys,
    crs=CRS_WGS84
)

# -----------------------------------------------
# 6. Save & summary
# -----------------------------------------------
print(f"\n📊 FINAL RESULTS:")
print(f"- Total hexagons: {len(gdf_hex)}")
print(f"- Measured (with segments): {(gdf_hex.segment_count > 0).sum()}")
print(f"- Predicted (regression): {(gdf_hex.segment_count == 0).sum()}")
print(f"- Walkability range: {gdf_hex.walkability_mean.min():.3f} to {gdf_hex.walkability_mean.max():.3f}")

# Save H3 grid
gdf_hex.to_file(os.path.join(output_step3_path, "step3_h3_walkability_final.gpkg"), driver="GPKG")
gdf_hex.to_parquet(f'{output_step3_path}/step3_h3_walkability_final.parquet')

print("✅ H3 walkability grid saved!")

# Show sample of the results
print(f"\nSample results:")
print(gdf_hex[['h3_id', 'walkability_mean', 'segment_count']].head(10))

Loaded 134070 segments with CRS: EPSG:2056
Calculated 134070 centroids in projected CRS
✅ Assigned H3 IDs to 134070 segments
✅ Created 2609 H3 hexagons with walkability data
Sample walkability values: 0    0.213567
1    0.393400
2    0.330283
3    0.381400
4    0.287419
Name: walkability_mean, dtype: float64
Walkability range: 0.182 to 1.000
✅ Added coordinates to 2609 H3 hexagons
Extended grid to 3054 hexagons (including neighbors)
Running regression: 2609 training points → 445 predictions
✅ Filled 445 missing values using linear regression

📊 FINAL RESULTS:
- Total hexagons: 3054
- Measured (with segments): 2609
- Predicted (regression): 445
- Walkability range: 0.182 to 1.000
✅ H3 walkability grid saved!

Sample results:
             h3_id  walkability_mean  segment_count
0  891f91ad3a7ffff          0.262294           48.0
1  891f91ae213ffff          0.355213           85.0
2  891f91ad56fffff          0.536441          157.0
3  891f91add13ffff          0.318554            0.0
4  891

**Export**

In [33]:
#save the file 
zones_mmt.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_mmt.gpkg"), driver="GPKG")
zones_mmt.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_mmt.parquet')